In [ ]:
import mne
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import hilbert, butter, filtfilt
import warnings
warnings.filterwarnings('ignore')

data_path = r"C:\Users\hiro2\Documents\EEG_project\data\MUSIN-G"
save_path = r"C:\Users\hiro2\Documents\EEG_project\data\connectivity"
os.makedirs(save_path, exist_ok=True)

MUSE_CH = [50, 26, 88, 7]
MUSE_CH_NAMES = ['AF7', 'AF8', 'TP9', 'TP10']
SF = 250
N_CH = 4

BANDS = {
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta':  (13, 30),
}

def bandpass(data, lo, hi, sf):
    nyq = sf / 2
    b, a = butter(4, [lo/nyq, hi/nyq], btype='band')
    return filtfilt(b, a, data)

def plv_matrix(data, sf, band):
    """Phase Locking Value行列"""
    n_ch = data.shape[0]
    plv_mat = np.zeros((n_ch, n_ch))
    lo, hi = band

    # 各チャンネルの位相を取得
    phases = []
    for ch_data in data:
        filtered = bandpass(ch_data, lo, hi, sf)
        analytic = hilbert(filtered)
        phases.append(np.angle(analytic))

    for i in range(n_ch):
        for j in range(n_ch):
            if i == j:
                plv_mat[i, j] = 1.0
                continue
            phase_diff = phases[i] - phases[j]
            plv = np.abs(np.mean(np.exp(1j * phase_diff)))
            plv_mat[i, j] = plv

    return plv_mat

for sub_id in range(1, 21):
    for ses_id in range(1, 13):
        set_path = os.path.join(
            data_path,
            f"sub-{sub_id:03d}",
            f"ses-{ses_id:02d}",
            "eeg",
            f"sub-{sub_id:03d}_ses-{ses_id:02d}_task-MusicListening_run-{ses_id}_eeg.set"
        )
        if not os.path.exists(set_path):
            continue

        try:
            raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
            raw.filter(1., 40., fir_window='hamming', verbose=False)
            data = raw.get_data()[MUSE_CH]

            conn_tensor = np.zeros((len(BANDS), N_CH, N_CH))
            for b_idx, (band_name, band_range) in enumerate(BANDS.items()):
                conn_tensor[b_idx] = plv_matrix(data, SF, band_range)

            fname = f"sub{sub_id:03d}_ses{ses_id:02d}.npy"
            np.save(os.path.join(save_path, fname), conn_tensor)

            print(f"✓ sub-{sub_id:03d} ses-{ses_id:02d} shape={conn_tensor.shape}")

        except Exception as e:
            print(f"✗ sub-{sub_id:03d} ses-{ses_id:02d}: {e}")

print(f"\n完了！{save_path} に保存")

# 可視化
sample = np.load(os.path.join(save_path, "sub001_ses01.npy"))
print("\nPLV値確認:")
print("theta:\n", np.round(sample[0], 3))
print("alpha:\n", np.round(sample[1], 3))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
band_names = list(BANDS.keys())

for i, ax in enumerate(axes):
    im = ax.imshow(sample[i], vmin=0, vmax=1,
                   cmap='hot', interpolation='nearest')
    ax.set_title(f'{band_names[i]} PLV')
    ax.set_xticks(range(N_CH))
    ax.set_yticks(range(N_CH))
    ax.set_xticklabels(MUSE_CH_NAMES)
    ax.set_yticklabels(MUSE_CH_NAMES)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig(r"C:\Users\hiro2\Documents\EEG_project\data\sample_connectivity.png",
            dpi=150)
print("PLV画像保存完了")